# Wrangling - property listing data

## Background
Analysis of property prices (to buy or to rent) in a region (city or state / country) is a good exercise to learn about simple regression analysis.  For a residential property (house), the price is determined roughly by the locality, size of land / house, age of the house and features within the house like bedrooms, bathrooms, living rooms, etc. In addition, factors like access to shopping centers, transportation, schools, distance to nearest city (CBD) also contribute on the property price.

This mini project integrates multiple datasets related to properties in Melbourne while finding and fixing issues with the data. These datasets comprise of the actual property data and related datasets of shopping centers, hospitals, supermarkets, transportation (metro trains) and geolocation.


## Data integration
**Objective** : Integrate 7 datasets into one based on given specifications.<br>
Datasets provided:
1. real_state.json - Property dataset (Set 1)
2. real_state.xml - Property Dataset (Set 2)
3. shopingcenters.html - Shopping Centers Dataset
4. supermarkets.json - Supermarkets Dataset
5. hospitals.xlsx
6. GTFS_Melbourne_Train_Information.zip - GTFS (General Transit Feed Specification) dataset for Melbourne Metro trains.
7. vic_suburb_boundary.zip - Shape file set for suburbs in Victoria, Australia. This comprises of four files within the zip.
  1. VIC_LOCALITY_POLYGON_shp.dbf
  2. VIC_LOCALITY_POLYGON_shp.prj
  3. VIC_LOCALITY_POLYGON_shp.shp
  4. VIC_LOCALITY_POLYGON_shp.shx

**Output dataset / file**<br>

CSV file (``property_merged.csv``) containing following fields
1. Record ID (Blank header)
2. property_id - Unique ID for property
3. lat - Latitude
4. lng - Longitude
5. addr_street - Street Address
6. suburb - Suburb
7. price
8. property_type - Property Type (from property dataset)
9. year - Year of sale
10. bedrooms - Number of bedrooms
11. bathrooms - Number of bathrooms
12. parking_space - Number of parking spaces
13. shopping_center_id - ID of nearest shopping center
14. distance_to_sc - Distance to nearest shopping center
15. train_station_id
16. distance_to_train_station
17. travel_min_to_CBD
18. transfer_flag
19. hospital_id
20. distance_to_hospital
21. supermarket_id
22. distance_to_supermaket

### Task 1 - Load property data
This involves loading the 2 datasets of property data, identifying and removing erroneous data records and merging into one dataset.

In [1]:
# Import libraries
import numpy as np
import json
import shapely
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# Load JSON file into "property1"
with open("../data/real_state.json", "r") as read_file:
    property1 = json.load(read_file)

In [3]:
# Check structure
print("property1 is a : " + str(type(property1)))

property1 is a : <class 'list'>


In [4]:
# Verify structure of property1. Each item within the list is a dictionary.
property1[0]

{'property_id': 53651.0,
 'lat': -37.82527542,
 'lng': 145.0058289,
 'addr_street': '69 Edinburgh Street',
 'price': 7875000.0,
 'property_type': 'house',
 'year': 2008.0,
 'bedrooms': 3.0,
 'bathrooms': 2.0,
 'parking_space': 1.0}

In [5]:
# load property1 list (of directory) into a dataframe
dfprop1 = pd.DataFrame.from_records(property1)

In [6]:
# Rearrange columns of dfprop1 - for better readability
dfprop1 = dfprop1[['property_id','lat','lng','addr_street',
                   'price','property_type','year','bedrooms',
                   'bathrooms','parking_space']]

In [7]:
# Check if there are any Null values in "property_id" column
dfprop1[dfprop1['property_id'].isnull()]

,property_id,lat,lng,addr_street,price,property_type,year,bedrooms,bathrooms,parking_space
41,NaN,NaN,NaN,None,NaN,None,NaN,NaN,NaN,NaN
323,NaN,NaN,NaN,None,NaN,None,NaN,NaN,NaN,NaN
557,NaN,NaN,NaN,None,NaN,None,NaN,NaN,NaN,NaN
659,NaN,NaN,NaN,None,NaN,None,NaN,NaN,NaN,NaN
706,NaN,NaN,NaN,None,NaN,None,NaN,NaN,NaN,NaN
871,NaN,NaN,NaN,None,NaN,None,NaN,NaN,NaN,NaN
941,NaN,NaN,NaN,None,NaN,None,NaN,NaN,NaN,NaN
987,NaN,NaN,NaN,None,NaN,None,NaN,NaN,NaN,NaN


In [8]:
len(dfprop1)

1010

In [9]:
# Check structure of dfprop1. There are totally 1002 samples (rows) having values. However, there are totally 1010 samples in the dataframe.
dfprop1.describe(include='all')

,property_id,lat,lng,addr_street,price,property_type,year,bedrooms,bathrooms,parking_space
count,1002.000000,1002.000000,1002.000000,1002,1.002000e+03,1002,1002.000000,1002.000000,1002.000000,1002.000000
unique,NaN,NaN,NaN,994,NaN,1,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,38 Woodburn Avenue,NaN,house,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,2,NaN,1002,NaN,NaN,NaN,NaN
mean,48712.836327,-37.804095,145.042389,NaN,9.109546e+06,NaN,2012.229541,3.198603,1.548902,1.493014
std,27695.243937,0.099350,0.130125,NaN,5.642116e+06,NaN,2.643146,0.766088,0.625869,0.778361
min,137.000000,-38.077602,144.715209,NaN,2.070000e+06,NaN,2005.000000,1.000000,1.000000,0.000000
25%,25302.500000,-37.863114,144.959803,NaN,5.200000e+06,NaN,2010.000000,3.000000,1.000000,1.000000
50%,48014.000000,-37.799359,145.033918,NaN,7.480000e+06,NaN,2012.000000,3.000000,1.000000,2.000000
75%,73020.250000,-37.731927,145.118375,NaN,1.123275e+07,NaN,2015.000000,4.000000,2.000000,2.000000


In [10]:
# Records which have NaN in property_id also do not have values in other columns. Remove such records.
dfprop1.dropna(how='all', inplace=True)

In [11]:
# Now check that blank records are removed.
len(dfprop1)

1002

In [12]:
# Check and verify that data types of columns are picked up properly. 
#    If there is any issue, need to correct them.
dfprop1.dtypes

property_id      float64
lat              float64
lng              float64
addr_street       object
price            float64
property_type     object
year             float64
bedrooms         float64
bathrooms        float64
parking_space    float64
dtype: object

In [13]:
# Create a new dataframe to get data from real_state.xml file. 
# The dataframe structure (columns) is based on analysis of XML file.
# The XML file has tags based on column values rather than row-based. Algorithm is below:
#     1. Create a blank dataframe with required columns (Child nodes under root in XML file - my term : Child-1).
#     2. "idx" represents row index in XML file, stored as tag for each node under Child-1 nodes.
#     3. Get tag and value for each child node under the Child-1 nodes.
#     4. If tag is not present in dataframe, create a new row.
#     5. If tag is present in dataframe "idx" column, add value under column for that row.
dfprop2 = pd.DataFrame(columns=['idx','bathrooms','price','bedrooms','parking_space','year',
                            'lat','lng','property_type','addr_street','property_id'])

In [ ]:
# Import cElementTree to read and traverse through the XML.
import xml.etree.cElementTree as ET

In [14]:
# Load file
tree = ET.parse('../data/real_state.xml')
root = tree.getroot()

# traverse through the XML file under root
# Immediate child nodes (Child-1) under root represent columns of dataframe.
# Each child node (Child-2) of "Child-1" nodes are cell values, for row represented by node tag.

for child1 in root:
    colname = child1.tag
    # Print column to show progress of data load
    print('Column: ' + colname + '  being updated')

    # traverse through cell value nodes (Child-2)
    for child2 in child1:
        rowid = child2.tag
        val = child2.text

        # If tag is available in idx column of dataframe, update "val" to appropriate cell.
        if (dfprop2['idx'] == rowid).any():
            dfprop2[colname][dfprop2['idx'] == rowid] = val
        # If tag is not available in idx column of dataframe, create new row with all nulls.
        else:
            dfblank = pd.DataFrame([[rowid,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan,'','',np.nan]],
                                   columns=['idx','bathrooms','price','bedrooms','parking_space','year',
                                            'lat','lng','property_type','addr_street','property_id'])
            dfprop2 = dfprop2.append(dfblank, ignore_index=True)
            dfprop2[colname][dfprop2['idx'] == rowid] = val


Column: bathrooms  being updated
Column: price  being updated
Column: bedrooms  being updated
Column: parking_space  being updated
Column: year  being updated
Column: lat  being updated
Column: lng  being updated
Column: property_type  being updated
Column: addr_street  being updated
Column: property_id  being updated


In [15]:
# Check structure of dfprop2. All columns are stored as objects, 
#     whereas only "idx", addr_street" and "property_type" are text and thus should be stored as objects.
dfprop2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1002 entries, 0 to 1001
Data columns (total 11 columns):
idx              1002 non-null object
bathrooms        1002 non-null object
price            1002 non-null object
bedrooms         1002 non-null object
parking_space    1002 non-null object
year             1002 non-null object
lat              1002 non-null object
lng              1002 non-null object
property_type    1002 non-null object
addr_street      1002 non-null object
property_id      1002 non-null object
dtypes: object(11)
memory usage: 86.2+ KB


In [16]:
# Try to cast individual column values to appropriate data type.
dfprop2.property_id = dfprop2.property_id.astype('float')
dfprop2.lat = dfprop2.lat.astype('float')
dfprop2.lng = dfprop2.lng.astype('float')
dfprop2.price = dfprop2.price.astype('float')
dfprop2.year = dfprop2.year.astype('float')
dfprop2.bedrooms = dfprop2.bedrooms.astype('float')
dfprop2.bathrooms = dfprop2.bathrooms.astype('float')
dfprop2.parking_space = dfprop2.parking_space.astype('float')

In [17]:
# Verify that column data types are converted properly.
dfprop2.dtypes

idx               object
bathrooms        float64
price            float64
bedrooms         float64
parking_space    float64
year             float64
lat              float64
lng              float64
property_type     object
addr_street       object
property_id      float64
dtype: object

In [18]:
# Now drop "idx" column as it is not required in final structure
dfprop2.drop(labels='idx', axis=1, inplace=True)

In [19]:
# Now, combine the 2 dataframes loaded from files
propdf = pd.concat([dfprop1, dfprop2], ignore_index=True, sort=False)

In [20]:
# Sort dataframe by property_id and check if values are proper
propdf.sort_values('property_id', inplace=True)

In [21]:
len(propdf)

2004

In [22]:
# Check which records have property_id null. These records have NaN in most columns. 
#   However, addr_street and property_type have a text "nan", thus cannot be identified as Null.
#   Modify these to None to remove these records. 
propdf[propdf['property_id'].isnull()]

,property_id,lat,lng,addr_street,price,property_type,year,bedrooms,bathrooms,parking_space
1368,NaN,NaN,NaN,nan,NaN,nan,NaN,NaN,NaN,NaN
1510,NaN,NaN,NaN,nan,NaN,nan,NaN,NaN,NaN,NaN
1523,NaN,NaN,NaN,nan,NaN,nan,NaN,NaN,NaN,NaN
1651,NaN,NaN,NaN,nan,NaN,nan,NaN,NaN,NaN,NaN
1654,NaN,NaN,NaN,nan,NaN,nan,NaN,NaN,NaN,NaN
1656,NaN,NaN,NaN,nan,NaN,nan,NaN,NaN,NaN,NaN
1726,NaN,NaN,NaN,nan,NaN,nan,NaN,NaN,NaN,NaN


In [23]:
propdf['addr_street'][propdf['addr_street'] == 'nan'] = None
propdf['property_type'][propdf['property_type'] == 'nan'] = None

/anaconda3/lib/python3.6/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.
/anaconda3/lib/python3.6/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  


In [24]:
# Now, drop rows where all columns are null
propdf.dropna(how='all', inplace=True)

In [25]:
# Reset Index to be sequential based on order of rows (based on property_id)
propdf.reset_index(inplace=True, drop=True)

In [26]:
# Check the length of the dataframe. It has 1997 rows (less 7 null rows removed)
len(propdf)

1997

### Task 2 - Add location data into property dataset

In [27]:
# Add the remaining columns with default values.
propdf['suburb']                    = ['not available'] * len(propdf.index)
propdf['shopping_center_id']        = ['not available'] * len(propdf.index)
propdf['distance_to_sc']            = [-0.01] * len(propdf.index)
propdf['train_station_id']          = [0] * len(propdf.index)
propdf['distance_to_train_station'] = [-0.01] * len(propdf.index)
propdf['travel_min_to_CBD']         = [-0.01] * len(propdf.index)
propdf['transfer_flag']             = [-1] * len(propdf.index) 
propdf['hospital_id']               = ['not available'] * len(propdf.index)
propdf['distance_to_hospital']      = [-0.01] * len(propdf.index)
propdf['supermarket_id']            = ['not available'] * len(propdf.index)
propdf['distance_to_supermaket']    = [-0.01] * len(propdf.index)

In [28]:
# import libraries required to derive suburb.
import shapefile
from shapely.geometry import Point
from shapely.geometry.polygon import Polygon
from time import time

In [29]:
# Load shapefile files. These are placed in project directory itself.
sf = shapefile.Reader('VIC_LOCALITY_POLYGON_shp')
recs = sf.records()
shapes = sf.shapes()

Nshp = len (shapes)

In [30]:
# Traverse through the propdf dataframe and get the suburb (shape).
# Algorithm:
# For each property in propdf dataframe,
#   1. Create a point "chkpnt" based on Longitude & Latitude of property
#   2. Traverse through the suburb dataset. For each,
#     2a. Calculate polygon of region based on points stored for each suburb
#     2b. Check if "chkpnt" is within the suburb polygon. 
#     2c. If yes, get suburb name and assign to propdf row.

##### TAKES LOT OF TIME TO RUN, PLEASE WAIT ####

tm = time() #store current epoch time to time the loop

for idx in propdf.index:
    
    chkpnt = Point(propdf.at[idx,'lng'], propdf.at[idx,'lat'])
    
    # temporary variable to flag when suburb is found
    bfound = False
    # Check each suburb if chkpint is within it.
    for nshp in range(Nshp):
        # If suburb has been found, break from loop
        if bfound:
            break;
            
        ptchs = []
        pts   = np.array(shapes[nshp].points)
        prt   = shapes[nshp].parts
        par   = list(prt) + [pts.shape[0]]

        for pij in range(len(prt)):
            poly = Polygon(pts[par[pij]:par[pij+1]])

            # If chkpnt is within the polygon, assign suburb name from "recs" to propdf row
            if poly.contains(chkpnt):
                propdf.at[idx, 'suburb'] = recs[nshp][6]
                bfound = True
                break;
    
    if (idx % 50 == 0): 
        print('calculated ' + str(idx) + ' records. Time elapsed: ' + str(time() - tm))
        # reset time for next check
        tm = time()


calculated 0 records. Time elapsed: 0.13309001922607422
calculated 50 records. Time elapsed: 8.417491912841797
calculated 100 records. Time elapsed: 5.357467889785767
calculated 150 records. Time elapsed: 13.558104991912842
calculated 200 records. Time elapsed: 15.49860405921936
calculated 250 records. Time elapsed: 25.75341296195984
calculated 300 records. Time elapsed: 21.504925966262817
calculated 350 records. Time elapsed: 11.701052188873291
calculated 400 records. Time elapsed: 16.525856018066406
calculated 450 records. Time elapsed: 26.266488790512085
calculated 500 records. Time elapsed: 12.95710802078247
calculated 550 records. Time elapsed: 11.333665132522583
calculated 600 records. Time elapsed: 16.05644392967224
calculated 650 records. Time elapsed: 17.982300996780396
calculated 700 records. Time elapsed: 21.094655990600586
calculated 750 records. Time elapsed: 12.96729588508606
calculated 800 records. Time elapsed: 24.64515709877014
calculated 850 records. Time elapsed: 4.0

In [31]:
# Check how many suburbs were missed because it could not be found based on lat & long
len(propdf[propdf['suburb'] == 'not available'])

0

### Task 3 - Add distance to nearest shopping center

In [32]:
# Let's define a function to calculate distance between two GPS coordinates in metres
# Source: https://stackoverflow.com/questions/19412462/getting-distance-between-two-points-based-on-latitude-longitude
import math

def distance(lat1, lon1, lat2, lon2):
######################################
#    Calculate the Haversine distance between 2 coordinates
#
#### Parameters ####
#    ----------
#    lat1 : Origin latitude
#    lon1 : Origin longitude
#    lat2 : Destination latitude
#    lon2 : Destination longitude
#
#### Returns ####
#    -------
#    distance_in_km : float
#
#### Examples ####
#    --------
#    # Origin: Munich
#    >>> lat1 = 48.1372
#    >>> lon1 = 11.5756
#    # Destination: Berlin
#    >>> lat2 = 52.5186
#    >>> lon2 = 13.4083
#    >>> round(distance(lat1, lon1, lat2, lon2), 1)
#    504.2
#########################################

    radius = 6378000  # earth's radius in metres as per "Specifications.pdf"

    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) * math.sin(dlat / 2) +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon / 2) * math.sin(dlon / 2))
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    d = radius * c
    return d

In [33]:
## Generic function to calculate distance between two GPS coordinates, each of which comes from a dataframe. ##
##### Parameters #####
#
#   srcDF: Source dataframe
#   srcId: ID (field) to update in source dataframe
#   srcDist: Distance field to update in source dataframe
#   srcLat, srcLng: Latitude & Longitude fields to use in source dataframe
#   refDF: Reference dataframe
#   refId: ID field to pick from reference dataframe
#   refLat, refLng: Latitude & Longitude fields to use from reference dataframe
#
##### Returns: none ##
#
##### uses ###########
#
# "distance" function to calculate distance
#
######################

def updLstDist(srcDF, srcId, srcDist, srcLat, srcLng, refDF, refId, refLat, refLng, _debug=False):
    # Below nested loop traverses through source dataframe and for each record,
    #   calculates distance with each record in reference dataframe. Whichever is smallest
    #   distance is stored against the source dataframe record.

    tm = time() #store current epoch time to time the loop
    tmtot = tm

    # for each property
    for idx in srcDF.index:
        # Store the latitude & longitude of source
        plat = srcDF.at[idx,srcLat]
        plng = srcDF.at[idx,srcLng]

        # instantiate distance
        shdist = 9999
        shp_id = 'not available'

        #for each reference dataframe record
        for idshp in refDF.index:
            #Store latitude, longitude & id of reference dataframe
            slat = refDF.at[idshp,refLat]
            slng = refDF.at[idshp,refLng]
            scID = refDF.at[idshp,refId]

            # Calculate distance from property to this reference record
            dist = distance(plat, plng, slat, slng)

            # if distance to this reference record is lesser than least till now, set this as nearest
            if (dist < shdist):
                shdist = dist
                shp_id = scID

        # If a reference record was found, assign to source
        if shdist != 9999:
            propdf.at[idx,srcId] = shp_id
            propdf.at[idx,srcDist] = shdist

        if _debug and (((idx + 1) % 500) == 0):
            print('calculated ' + str(idx + 1) + ' records. Time elapsed: ' + str(time() - tm))
            # reset time for next check
            tm = time()

    if _debug: 
        print('Total calculated ' + str(idx + 1) + ' records. Total time elapsed: ' + str(time() - tmtot))

In [34]:
# Load shopping centers from shopincenters.html to a dataframe. 
# This loads file as a list, of which first item is a dataframe. Thus, get the first item only.
with open("../data/shopingcenters.html", "r") as read_file:
    dfshop = pd.read_html(read_file, index_col=0)[0]

In [35]:
dfshop.head()

,sc_id,lat,lng
0,SC_001,-37.767915,145.041790
1,SC_002,-37.819375,145.171472
2,SC_003,-37.971131,145.089065
3,SC_004,-35.280406,149.132550
4,SC_005,-37.574572,144.920451


In [36]:
# Check if there are any nulls. There are no Nulls
dfshop.isnull().sum()

sc_id    0
lat      0
lng      0
dtype: int64

In [37]:
updLstDist(propdf, 'shopping_center_id', 'distance_to_sc', 'lat', 'lng',
           dfshop, 'sc_id', 'lat', 'lng', True)

calculated 500 records. Time elapsed: 0.8543219566345215
calculated 1000 records. Time elapsed: 0.8440980911254883
calculated 1500 records. Time elapsed: 0.8588910102844238
Total calculated 1997 records. Total time elapsed: 3.4019949436187744


In [38]:
# Check how many shopping centers were missed because it could not be found based on lat & long
len(propdf[propdf['shopping_center_id'] == 'not available'])

0

In [39]:
# Check how many shopping center distance was not updated because it could not be found based on lat & long
propdf[propdf['distance_to_sc'] <= 0.00]

,property_id,lat,lng,addr_street,price,property_type,year,bedrooms,bathrooms,parking_space,...,shopping_center_id,distance_to_sc,train_station_id,distance_to_train_station,travel_min_to_CBD,transfer_flag,hospital_id,distance_to_hospital,supermarket_id,distance_to_supermaket


### Task 4 - Add distance to nearest supermarket

In [40]:
# Load Shopping centers from "supermarkets.json".
# This loads file as a list, each item containing a dictionary. Conver this to a dataframe.
with open("../data/supermarkets.json", "r") as read_file:
    supr = json.load(read_file)

In [41]:
# load list (of dictionary) into a dataframe
dfsupr = pd.DataFrame.from_records(supr)
dfsupr.head()

,id,lat,lng,type
0,S_001,-37.883978,144.735287,Woolworths
1,S_002,-41.161591,147.514797,Woolworths
2,S_003,-37.984078,145.077167,Woolworths
3,S_004,-37.707023,144.938740,Woolworths
4,S_005,-37.597670,144.938413,Woolworths


In [42]:
updLstDist(propdf, 'supermarket_id', 'distance_to_supermaket', 'lat', 'lng',
           dfsupr, 'id', 'lat', 'lng', True)

calculated 500 records. Time elapsed: 1.697558879852295
calculated 1000 records. Time elapsed: 1.6698079109191895
calculated 1500 records. Time elapsed: 1.6908378601074219
Total calculated 1997 records. Total time elapsed: 6.738217115402222


In [43]:
# Check how many supermarkets were missed because it could not be found based on lat & long
print(len(propdf[propdf['supermarket_id'] == 'not available']))
print(propdf[propdf['distance_to_supermaket'] <= 0])

0
Empty DataFrame
Columns: [property_id, lat, lng, addr_street, price, property_type, year, bedrooms, bathrooms, parking_space, suburb, shopping_center_id, distance_to_sc, train_station_id, distance_to_train_station, travel_min_to_CBD, transfer_flag, hospital_id, distance_to_hospital, supermarket_id, distance_to_supermaket]
Index: []

[0 rows x 21 columns]


### Task 5 - Add distance to nearest hospital

In [44]:
# Read Hospitals.xlsx
dfhosp = pd.read_excel("../data/hospitals.xlsx")

In [45]:
dfhosp.head()

,id,lat,lng,name
0,hospital_001,-37.990622,145.072836,Como Private Hospital
1,hospital_002,-37.855469,145.268183,Mountain District Private Hospital
2,hospital_003,-37.792230,144.889128,Western Hospital
3,hospital_004,-37.756042,145.061896,Mercy Hospital for Women
4,hospital_005,-37.760623,144.815624,Sunshine Hospital


In [46]:
#  Load nearest hopital for each property into "propdf"
updLstDist(propdf, 'hospital_id', 'distance_to_hospital', 'lat', 'lng',
           dfhosp, 'id', 'lat', 'lng', True)

# Check how many hospitals were missed because it could not be found based on lat & long
print(len(propdf[propdf['hospital_id'] == 'not available']))
print(propdf[propdf['distance_to_hospital'] <= 0.00])

calculated 500 records. Time elapsed: 1.4041659832000732
calculated 1000 records. Time elapsed: 1.4002931118011475
calculated 1500 records. Time elapsed: 1.422550916671753
Total calculated 1997 records. Total time elapsed: 5.618445873260498
0
Empty DataFrame
Columns: [property_id, lat, lng, addr_street, price, property_type, year, bedrooms, bathrooms, parking_space, suburb, shopping_center_id, distance_to_sc, train_station_id, distance_to_train_station, travel_min_to_CBD, transfer_flag, hospital_id, distance_to_hospital, supermarket_id, distance_to_supermaket]
Index: []

[0 rows x 21 columns]


### Task 6 - Identify nearest train station for each property and calculate average commute time to CBD (Flinders Street)

#### Load train information
**_Available datasets_**:  
**(GTFS - Melbourne Train Information)**
1. agency.txt - Information of service provider
2. calendar_dates.txt - List of dates on which regular service is either disrupted or additional service is provided
3. calendar.txt - List of (train) service codes provided with details of day of week when service is provided.
4. routes.txt - List of routes on which trains is run, with details of agency, type of service (metro, regional, etc)
5. shapes.txt - Shape files of the routes taken by the train services (used to plot train routes)
6. stop_times.txt - list of arrival and departure times of each trip along the stops that each stop at.
7. stops.txt - List of stops for the trains.
8. trips.txt - List of trips (runs) that the trains operate on.

#### Requirement
- Identify nearest train station to each property and (Euclidean) distance between train station and property. *Note: restrict to train stations which provide train stations during weekday*
- Calculate time to travel to CBD "Flinders Street from nearest train station on weekday starting between 7 AM and 9 AM.
- Calculate if there is a direct train during the above period or a transfer is needed.

#### Algorithm
1. Load records from Calendar.txt where service runs on weekdays
2. Load trips data and Merge trips with weekday services to get trip details of train which run on weekdays.
3. Load list of unique stops along the trip route of weekday trips.
4. Find the nearest stop to each property from above stop list.
5. Calculate distance from property to nearest train station.
6. Get trips where train (train1) starts from stop in dfWkstops between 7 AM and 9 AM.
7. Get last stop (last_1) of each trip.
    7.a. If stop_id (last_1) is Flinders St (19854), this is a direct train (transfer_flag = 0)
    7.b. If stop_id (last_1) is not Flinders St, transfer_flag = 1.
8. Where transfer_flag = 1, find first trip which arrives after train_1 arrives at last_1 and where Flinder's St is a stop.
9. Set the arrival time at Flinder St as end time of journey
7. Calculate time for travel based on departure from source stop to Flinders St.

In [47]:
# Load records from Calendar.txt where service runs on weekdays
dfCalendar = pd.read_csv('../data/calendar.txt')

#Load trips.txt file
dftrips = pd.read_csv('../data/trips.txt')

In [48]:
# Verify the data loaded
dfCalendar.head(1)

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,T2,0,0,0,0,0,1,0,20151009,20151011


In [50]:
dftrips.head(1)

,route_id,service_id,trip_id,shape_id,trip_headsign,direction_id
0,2-ALM-F-mjp-1,T0,17067982.T0.2-ALM-F-mjp-1.1.H,2-ALM-F-mjp-1.1.H,City (Flinders Street),0


In [51]:
# We are interested in trains operating in weekday for our calculations
filt = ((dfCalendar['monday'] == 1) | (dfCalendar['tuesday'] == 1) | (dfCalendar['wednesday'] == 1) 
        | (dfCalendar['thursday'] == 1) | (dfCalendar['friday'] == 1))
dfcal = dfCalendar[filt]

In [52]:
# Merge trips with weekday services to get trip details of train which run on weekdays 
dfWktrips = pd.merge(dftrips, dfcal, how='inner', on='service_id')

# Drop unwanted fields
dfWktrips.drop(labels=['shape_id','saturday','sunday'], axis=1, inplace=True)

# Only consider the trips which lead to the city.
dfWktrips = dfWktrips[dfWktrips['direction_id'] == 0]

In [53]:
dfWktrips.describe(include='all')

,route_id,service_id,trip_id,trip_headsign,direction_id,monday,tuesday,wednesday,thursday,friday,start_date,end_date
count,6415,6415,6415,6415,6415.0,6415.000000,6415.000000,6415.000000,6415.000000,6415.000000,6.415000e+03,6.415000e+03
unique,80,13,6415,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,2-SYM-B-mjp-1,T6_1,17191172.T6.2-HBG-D-mjp-1.5.H,City (Flinders Street),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,221,1313,1,4934,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,0.0,0.585814,0.585814,0.585814,0.585814,0.590335,2.015102e+07,2.015104e+07
std,NaN,NaN,NaN,NaN,0.0,0.492619,0.492619,0.492619,0.492619,0.491810,4.888536e+00,4.081205e+01
min,NaN,NaN,NaN,NaN,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,2.015101e+07,2.015101e+07
25%,NaN,NaN,NaN,NaN,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,2.015101e+07,2.015102e+07
50%,NaN,NaN,NaN,NaN,0.0,1.000000,1.000000,1.000000,1.000000,1.000000,2.015102e+07,2.015102e+07
75%,NaN,NaN,NaN,NaN,0.0,1.000000,1.000000,1.000000,1.000000,1.000000,2.015102e+07,2.015102e+07


In [54]:
# Load stop_times.txt
dfstoptimes = pd.read_csv('../data/stop_times.txt')

dfstoptimes.describe(include='all')

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled
count,390305,390305,390305,390305.000000,390305.000000,0.0,390305.0,390305.0,390217.000000
unique,23809,1350,1350,NaN,NaN,NaN,NaN,NaN,NaN
top,17183649.T5.2-BEL-C-mjp-1.22.R,17:10:00,17:10:00,NaN,NaN,NaN,NaN,NaN,NaN
freq,32,577,577,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,20510.975942,11.344659,NaN,0.0,0.0,16494.255519
std,NaN,NaN,NaN,3607.289276,7.554997,NaN,0.0,0.0,12492.655423
min,NaN,NaN,NaN,15351.000000,1.000000,NaN,0.0,0.0,0.000000
25%,NaN,NaN,NaN,19870.000000,5.000000,NaN,0.0,0.0,6673.537679
50%,NaN,NaN,NaN,19923.000000,10.000000,NaN,0.0,0.0,14252.835693
75%,NaN,NaN,NaN,19979.000000,17.000000,NaN,0.0,0.0,23921.432129


In [55]:
dfstoptimes.drop(labels=['stop_headsign','pickup_type','drop_off_type','shape_dist_traveled'],
                 axis=1, inplace=True)

dfstoptimes.sort_values(['trip_id', 'stop_sequence'], inplace=True)
dfstoptimes.head(2)

,trip_id,arrival_time,departure_time,stop_id,stop_sequence
109511,17067050.T0.2-EPP-F-mjp-1.8.R,04:24:00,04:24:00,45795,1
109512,17067050.T0.2-EPP-F-mjp-1.8.R,04:27:00,04:27:00,20005,2


In [56]:
# Get all the stops which have trains running on weekdays
dfWkstops = pd.merge(dfstoptimes[['stop_id','trip_id']], dfWktrips[['trip_id']], 
                     how='inner', on='trip_id')
dfWkstops.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 107862 entries, 0 to 107861
Data columns (total 2 columns):
stop_id    107862 non-null int64
trip_id    107862 non-null object
dtypes: int64(1), object(1)
memory usage: 2.5+ MB


In [57]:
# Drop Trips as we dont need it
dfWkstops.drop(labels=['trip_id'], axis=1, inplace=True)

# Drop duplicates so that we have an unique list
dfWkstops.drop_duplicates(inplace=True)

dfWkstops.reset_index(drop=True, inplace=True)

# Check dataset now
dfWkstops.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 216 entries, 0 to 215
Data columns (total 1 columns):
stop_id    216 non-null int64
dtypes: int64(1)
memory usage: 1.8 KB


In [58]:
# Get all stops from dataset
dfstops = pd.read_csv('../data/stops.txt')

dfstops.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 5 columns):
stop_id            218 non-null int64
stop_name          218 non-null object
stop_short_name    218 non-null object
stop_lat           218 non-null float64
stop_lon           218 non-null float64
dtypes: float64(2), int64(1), object(2)
memory usage: 8.6+ KB


In [59]:
# merge stops with weekday stops to check which of them have trains on weekdays
dfstops = pd.merge(dfstops, dfWkstops, how='inner', on='stop_id')

# Update nearest train station and distance based on trains running on weekdays
updLstDist(propdf, 'train_station_id', 'distance_to_train_station', 'lat', 'lng',
           dfstops, 'stop_id', 'stop_lat', 'stop_lon', True)

calculated 500 records. Time elapsed: 1.5186619758605957
calculated 1000 records. Time elapsed: 1.5446510314941406
calculated 1500 records. Time elapsed: 1.5166029930114746
Total calculated 1997 records. Total time elapsed: 6.090464115142822


In [60]:
#Verify the first few records with columns updated recently (train)
propdf[['property_id','addr_street','lat','lng','train_station_id','distance_to_train_station']].head(1)

,property_id,addr_street,lat,lng,train_station_id,distance_to_train_station
0,137.0,10 Batman Street,-37.799381,144.889297,20024,396.829197


In [61]:
# Check how many records do not have train station updated
propdf[propdf['train_station_id'] <= 0]

,property_id,lat,lng,addr_street,price,property_type,year,bedrooms,bathrooms,parking_space,...,shopping_center_id,distance_to_sc,train_station_id,distance_to_train_station,travel_min_to_CBD,transfer_flag,hospital_id,distance_to_hospital,supermarket_id,distance_to_supermaket


In [62]:
# Convert arrival time and departure time into seconds from 12:00:00 AM
dfstoptimes['arrival_time_secs'] = dfstoptimes['arrival_time'].apply(
    lambda x: (int(x.split(':')[0]) * 3600) + 
    (int(x.split(':')[1]) * 60) + 
    (int(x.split(':')[2])))

dfstoptimes['departure_time_secs'] = dfstoptimes['departure_time'].apply(
    lambda x: (int(x.split(':')[0]) * 3600) + 
    (int(x.split(':')[1]) * 60) + 
    (int(x.split(':')[2])))

# Verify Dataset
dfstoptimes.head(1)

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,arrival_time_secs,departure_time_secs
109511,17067050.T0.2-EPP-F-mjp-1.8.R,04:24:00,04:24:00,45795,1,15840,15840


In [63]:
# Algorithm to get travel time to CBD (Flinders) & transfer flag
# 1. Join dfstoptimes with dfWkstops to get trips where 
#     train starts from stop in dfWkstops between 7 AM and 9 AM.
#     This would be train trips for each stop.
# 1.a. Data fields to consider: trip_id, departure_time_secs, stop_sequence
# 2. Get last stop (1) of each trip - trip_id, arrival_time_secs, stop_id
# 3. If stop_id (last_1) is Flinders St (19854), this is a direct train (transfer_flag = 0)
# 4. If stop_id (last_1) is not Flinders St, transfer_flag = 1.
# 5. Where transfer_flag = 1, find first record in dfstoptimes where 
#     departure_time_secs is greater than arrival time of train to reach there.
#     Fields to consider (train2): trip_id, departure_time_secs, stop_sequence
# 6. for train2 trip, get Flinders St stop record - arrival_time_secs.
# 7. Difference between departure_time_secs (source stop) to arrival_time_secs (Flinders St)
#     is total travel time to CBD.
#################
# 7:00:00 AM = 25200
# 9:00:00 AM = 32400

# Get smaller dataset of trip stops where departure is between 7 AM to 9 AM
dfWktravel = dfstoptimes[dfstoptimes['departure_time_secs'] >= 25200]
dfWktravel = dfWktravel[dfWktravel['departure_time_secs'] <= 32400]

# Merge trip stops with weekday trips
dfWktravel = pd.merge(dfWktravel,dfWkstops, on='stop_id', how='inner')

# Drop unwanted fields
dfWktravel.drop(labels=['arrival_time','arrival_time_secs'], axis=1, inplace=True)

# Rename fields to make it meaningful
dfWktravel.rename(columns = {'departure_time':'start_at', 'departure_time_secs':'start_at_secs'},inplace = True)

# Create fields to calculate and set default values
dfWktravel['last_stop_1']   = [-1] * len(dfWktravel.index) 
dfWktravel['end_at_1']      = [-1] * len(dfWktravel.index) 
dfWktravel['last_stop_2']   = [-1] * len(dfWktravel.index) 
dfWktravel['end_at_2']      = [-1] * len(dfWktravel.index)
dfWktravel['transfer_flag'] = [-1] * len(dfWktravel.index)
dfWktravel['travel_time']   = [0] * len(dfWktravel.index)

In [64]:
# Main code to calculate "transfer_flag" and "travel time to CBD"
istartstop = -1
tm = time() #store current epoch time to time the loop
tmtot = tm

# for each record in dfWktravel
for idx1 in dfWktravel.index:
    iStStop  = dfWktravel.at[idx1,'stop_id']
    sTripId  = dfWktravel.at[idx1,'trip_id']
    iStopSeq = dfWktravel.at[idx1,'stop_sequence']
    
#    print('Stop: ' + str(iStStop) + ' Trip: ' + sTripId + ' StopSeq: ' + str(iStopSeq))
    
    if (iStStop == 19854):
        dfWktravel.at[idx1,'last_stop_2'] = dfWktravel.at[idx1,'stop_id']
        dfWktravel.at[idx1,'end_at_2']    = dfWktravel.at[idx1,'start_at_secs']
    else:
        # Check if Flinders St is a stop in dfstoptimes for this trip
        filt1 = ((dfstoptimes['trip_id'] == sTripId) &
                 (dfstoptimes['stop_id'] == 19854) & 
                (dfstoptimes['stop_sequence'] >= iStopSeq)) # Flinders St
        dflaststop = dfstoptimes[filt1]

        # If there is a stop at Flinders St for this trip
        if len(dflaststop) > 0:
#            print(dflaststop)
            lsidx = dflaststop.index[0]
            dfWktravel.at[idx1,'last_stop_2']   = dflaststop.at[lsidx,'stop_id']
            dfWktravel.at[idx1,'end_at_2']      = dflaststop.at[lsidx,'arrival_time_secs']
            dfWktravel.at[idx1,'transfer_flag'] = 0
        else:
#            print(dflaststop)
            dfWktravel.at[idx1,'transfer_flag'] = 1
            
            filt2 = ((dfstoptimes['trip_id'] == sTripId) &
                    (dfstoptimes['stop_sequence'] >= iStopSeq)) # Flinders St
            dftransfstop = dfstoptimes[filt2]
#            print('dftransfstop')
#            print(dftransfstop)
#            dftransfstop.sort_values(['stop_sequence'], inplace=True)
            
            ilastseq = dftransfstop.index.max()
#            print(ilastseq)
            laststop1 = dftransfstop.at[ilastseq,'stop_id']
            nexttrainafter = dftransfstop.at[ilastseq,'arrival_time_secs']
            dfWktravel.at[idx1,'last_stop_1']   = laststop1
            dfWktravel.at[idx1,'end_at_1']      = nexttrainafter
            
            filt3 = ((dfstoptimes['stop_id'] == laststop1) & 
                     dfstoptimes['arrival_time_secs'] > nexttrainafter)
            dftransftrain = dfstoptimes[filt3]
            dftransftrain.sort_values(['arrival_time_secs','trip_id'], inplace=True)
            
            itransCtx = 1 # Counter, to stop at 50 trains
            
            for idx2 in dftransftrain.index:
                stranstrip   = dftransftrain.at[idx2,'trip_id']
                itranStopSeq = dftransftrain.at[idx2,'stop_sequence']
                
                filt3 = ((dfstoptimes['trip_id'] == stranstrip) &
                        (dfstoptimes['stop_id'] == 19854))
                
                dflaststop2 = dfstoptimes[filt3]
                
                if len(dflaststop2) > 0:
                    lsidx2 = dflaststop2.index.min()
                    dfWktravel.at[idx1,'last_stop_2']   = dflaststop2.at[lsidx2,'stop_id']
                    dfWktravel.at[idx1,'end_at_2']      = dflaststop2.at[lsidx2,'arrival_time_secs']
                    break
                
                if itransCtx > 50:
                    print('Could not find second leg of travel from stop: ' + str(iStStop))
                    print('Transfer at: ' + str(laststop1) + ' at time : ' + str(nexttrainafter))

                    dfWktravel.at[idx1,'last_stop_2']   = -999
                    dfWktravel.at[idx1,'end_at_2']      = -999
                    break
                
                itransCtx = itransCtx + 1
        
    # Now check if we were able to find a route to Flinder's St
    if (dfWktravel.at[idx1,'last_stop_2'] == 19854):
        dfWktravel.at[idx1,'travel_time'] = round((dfWktravel.at[idx1,'end_at_2'] - 
                                                   dfWktravel.at[idx1,'start_at_secs']) / 60)
    else:
        dfWktravel.at[idx1,'travel_time'] = 0
    
    if idx1 % 500 == 0:
        print('processed ' + str(idx1) + ' records. Time elapsed till now: ' + str(time() - tm))

print('done')

processed 0 records. Time elapsed till now: 0.02824687957763672


/anaconda3/lib/python3.6/site-packages/ipykernel_launcher.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy


processed 500 records. Time elapsed till now: 35.007954835891724
processed 1000 records. Time elapsed till now: 68.00818586349487
processed 1500 records. Time elapsed till now: 99.98554968833923
processed 2000 records. Time elapsed till now: 108.87229084968567
processed 2500 records. Time elapsed till now: 108.9004738330841
processed 3000 records. Time elapsed till now: 108.92873978614807
processed 3500 records. Time elapsed till now: 108.95676398277283
processed 4000 records. Time elapsed till now: 108.9848039150238
processed 4500 records. Time elapsed till now: 135.76246881484985
processed 5000 records. Time elapsed till now: 164.18324303627014
processed 5500 records. Time elapsed till now: 204.29639887809753
processed 6000 records. Time elapsed till now: 234.4904589653015
processed 6500 records. Time elapsed till now: 264.4616458415985
processed 7000 records. Time elapsed till now: 284.11990189552307
processed 7500 records. Time elapsed till now: 310.8032829761505
processed 8000 rec

In [65]:
# Create a new dataset with mean values of travel time for each Stop
func1 = {'travel_time': 'mean'}
dfStopAvg = dfWktravel.groupby(by=['stop_id', 'transfer_flag']).agg(func1).reset_index()
dfStopAvg.sort_values(by=['stop_id','transfer_flag'])
dfStopAvg.drop_duplicates('stop_id',inplace=True)

dfStopAvg.rename(columns = {'stop_id' : 'train_station_id', 'travel_time' : 'travel_min_to_CBD'},inplace = True)

# Set value for Flinders St. There will be no trips for this record.
idx = dfStopAvg[dfStopAvg['train_station_id'] == 19854].index.min()
dfStopAvg.at[idx,'transfer_flag'] = 0
dfStopAvg.at[idx,'travel_min_to_CBD'] = 0

In [66]:
# Verify dataset
dfStopAvg.describe()

,train_station_id,transfer_flag,travel_min_to_CBD
count,216.000000,216.000000,216.000000
mean,20684.175926,0.041667,30.502948
std,4355.579530,0.200290,16.792676
min,15351.000000,0.000000,0.000000
25%,19878.750000,0.000000,19.007950
50%,19932.500000,0.000000,29.607143
75%,19989.250000,0.000000,39.655107
max,46468.000000,1.000000,80.250000


In [67]:
#Merge train journey data into Property Dataset
propdf.drop(labels=['travel_min_to_CBD','transfer_flag'], axis=1, inplace=True)
propdf = pd.merge(propdf,dfStopAvg, on='train_station_id', how='left')

In [70]:
# Verify the structure of property dataset
propdf.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1997 entries, 0 to 1996
Data columns (total 21 columns):
property_id                  1997 non-null float64
lat                          1997 non-null float64
lng                          1997 non-null float64
addr_street                  1997 non-null object
price                        1997 non-null float64
property_type                1997 non-null object
year                         1997 non-null float64
bedrooms                     1997 non-null float64
bathrooms                    1997 non-null float64
parking_space                1997 non-null float64
suburb                       1997 non-null object
shopping_center_id           1997 non-null object
distance_to_sc               1997 non-null float64
train_station_id             1997 non-null int64
distance_to_train_station    1997 non-null float64
hospital_id                  1997 non-null object
distance_to_hospital         1997 non-null float64
supermarket_id               1997

In [69]:
#Save final data to CSV.
propdf.to_csv('property_merged.csv')